# LangChain Tool Calling with OpenAI
Ví dụ định nghĩa, bind và gọi tools bằng LangChain.

In [22]:
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import (
    ArxivLoader,
    WebBaseLoader,
    WikipediaLoader,
)
import arxiv
import requests
import time
import wikipedia
import xml.etree.ElementTree as ET
from urllib.parse import quote
from langchain_core.documents import Document

load_dotenv()

True

In [23]:
# wikipedia 1.4.0 builds an HTTP API URL; Wikimedia requires HTTPS.
# Wrap set_lang once (re-run safe) so it always forces the API_URL to HTTPS.
_original_set_lang = getattr(wikipedia, "_original_set_lang", wikipedia.set_lang)
wikipedia._original_set_lang = _original_set_lang

def set_wikipedia_lang_https(prefix: str) -> None:
    _original_set_lang(prefix)
    wikipedia.wikipedia.API_URL = wikipedia.wikipedia.API_URL.replace("http://", "https://")

wikipedia.set_lang = set_wikipedia_lang_https
wikipedia.set_user_agent("LangChainToolCallingNotebook/1.0 (educational use)")


# arxiv 1.x also defaults to HTTP, which is blocked by many environments.
arxiv.Client.query_url_format = arxiv.Client.query_url_format.replace("http://", "https://", 1)

## Define tools

In [24]:
from langchain_core.tools import tool

@tool
def reverse_string(input_string: str) -> str:
    """Reverse a string """
    return input_string[::-1]

reversed_result = reverse_string.invoke({"input_string": "Hello VuVau"})

print(reversed_result)

uaVuV olleH


In [25]:
def format_documents(documents, max_chars: int = 4000) -> str:
    """Convert loaded documents into compact text for the model."""
    if not documents:
        return "Không tìm thấy tài liệu."
    sections = []
    for index, document in enumerate(documents, start=1):
        title = document.metadata.get("title", f"Tài liệu {index}")
        source = document.metadata.get("source", document.metadata.get("entry_id", "Không rõ nguồn"))
        sections.append(f"Tiêu đề: {title}\nNguồn: {source}\nNội dung:\n{document.page_content[:max_chars]}")
    return "\n\n---\n\n".join(sections)

def load_with_retry(loader_factory, attempts: int = 3, delay: float = 1.0):
    """Retry transient loader/network failures, then raise the last error."""
    last_error = None
    for attempt in range(attempts):
        try:
            documents = loader_factory().load()
            if documents:
                return documents
        except Exception as exc:
            last_error = exc
        if attempt < attempts - 1:
            time.sleep(delay * (attempt + 1))
    if last_error is not None:
        raise last_error
    return []

@tool
def search_wikipedia(query: str, top_k: int = 2) -> str:
    """Tìm thông tin bách khoa trên Wikipedia."""
    try:
        # Call WikipediaLoader
        documents = load_with_retry(lambda: WikipediaLoader(query=query, load_max_docs=top_k, lang="vi"))
        return format_documents(documents)
        

    except Exception as loader_exc:
        # Fallback for wikipedia 1.4.0/API responses that are not valid JSON.
        try:
            # Avoid another API request (and possible 429): load the likely article URL directly.
            article_url = f"https://vi.wikipedia.org/wiki/{quote(query.strip().replace(' ', '_'))}"
            return format_documents(WebBaseLoader(web_paths=(article_url,), requests_kwargs={"timeout": 20}).load())
        except Exception as fallback_exc:
            return (f"Wikipedia hiện không truy cập được: primary={type(loader_exc).__name__}: {loader_exc}; "
                    f"fallback={type(fallback_exc).__name__}: {fallback_exc}")

@tool
def search_arxiv(query: str, top_k: int = 2) -> str:
    """Tìm các bài nghiên cứu khoa học trên arXiv."""
    try:
        # Call ArxivLoader
        documents = load_with_retry(lambda: ArxivLoader(query=query, load_max_docs=top_k, load_all_avilable_meta=True))

    except Exception as loader_exc:
        # Fallback when arxiv 1.x/feedparser cannot expose the HTTP status.
        try:
            response = requests.get(
                "https://export.arxiv.org/api/query",
                params={"search_query": query, "start": 0, "max_results": top_k},
                headers={"User-Agent": "LangChainToolCallingNotebook/1.0 (educational use)"},
                timeout=30,
            )
            response.raise_for_status()
            root = ET.fromstring(response.content)
            ns = {"atom": "http://www.w3.org/2005/Atom"}
            documents = []
            for entry in root.findall("atom:entry", ns):
                title = " ".join((entry.findtext("atom:title", default="Untitled", namespaces=ns)).split())
                summary = " ".join((entry.findtext("atom:summary", default="", namespaces=ns)).split())
                source = entry.findtext("atom:id", default="", namespaces=ns)
                documents.append(Document(page_content=summary, metadata={"title": title, "source": source}))
            return format_documents(documents)
        except Exception as fallback_exc:
            return (f"arXiv hiện không truy cập được: primary={type(loader_exc).__name__}: {loader_exc}; "
                    f"fallback={type(fallback_exc).__name__}: {fallback_exc}")

@tool
def load_web_page(url: str) -> str:
    """Tải và đọc nội dung từ một URL cụ thể."""
    try:
        # Call WebBaseLoader
        return format_documents(WebBaseLoader(web_paths=(url,), requests_kwargs={"timeout": 15}).load())

    except Exception as exc:
        return f"Không tải được trang web: {type(exc).__name__}: {exc}"

# print(search_wikipedia.invoke({"query": "Trí tuệ nhân tạo", "top_k": 1})[:500])

In [26]:
# Multiply tool
@tool
def multiply(a: float, b: float) -> float:
    """ Multiply two numbers. """
    return a * b

multipy = multiply.invoke({"a": 2, "b": 3})

print(multipy)

6.0


## OpenAI model

In [27]:
import os
llm = ChatOpenAI(model="gpt-5-mini", temperature=0, api_key=os.getenv("OPENAI_API_KEY"))

## Binding and executing tools

In [28]:
tools = [search_wikipedia, search_arxiv, load_web_page, reverse_string, multiply]

model_with_tools = llm.bind_tools(tools)

In [33]:
response = model_with_tools.invoke("Cho tôi thông tin về thủ đô Hà Nội của nước Việt Nam")

for tool_call in response.tool_calls:
    print(tool_call)

## Additional tests
Các kiểm thử dưới đây xác nhận tool cơ bản, loader và OpenAI tool calling.

In [34]:
# Run every loader and display its full result
loader_cases = {
    "wikipedia": (search_wikipedia, {"query": "Trí tuệ nhân tạo", "top_k": 1}),
    "arxiv": (search_arxiv, {"query": "cat:cs.AI AND ti:agent", "top_k": 1}),
    "web": (load_web_page, {"url": "https://example.com"}),
}
loader_results = {}
for name, (loader_tool, arguments) in loader_cases.items():
    output = loader_tool.invoke(arguments)
    loader_results[name] = output
    print(f"\n{'=' * 20} {name.upper()} {'=' * 20}")
    print(output)

loader_results


==================== WIKIPEDIA ====================
Tiêu đề: Trí tuệ nhân tạo
Nguồn: https://vi.wikipedia.org/wiki/Tr%C3%AD_tu%E1%BB%87_nh%C3%A2n_t%E1%BA%A1o
Nội dung:
Trí tuệ nhân tạo (tiếng Anh: Artificial intelligence, viết tắt là AI) là khả năng của các hệ thống máy tính thực hiện các nhiệm vụ liên quan đến trí thông minh của con người, như học tập, suy luận, giải quyết vấn đề, nhận thức và đưa ra quyết định. Đây là một lĩnh vực nghiên cứu thuộc khoa học máy tính tập trung phát triển và nghiên cứu các phương pháp và phần mềm giúp máy móc có thể nhận thức môi trường xung quanh, cũng như sử dụng học tập và trí tuệ để thực hiện hành động nhằm tối đa hóa khả năng đạt được các mục tiêu đã định.
Các ứng dụng nổi bật của AI bao gồm công cụ tìm kiếm web tiên tiến (ví dụ: Google Tìm kiếm); hệ thống đề xuất (được sử dụng bởi YouTube, Amazon và Netflix); trợ lý ảo (ví dụ: Trợ lý Google, Siri và Alexa); xe tự lái (ví dụ: Waymo); công cụ sáng tạo và nội dung tạo sinh (ví dụ: mô hình ngôn ngữ v

{'wikipedia': 'Tiêu đề: Trí tuệ nhân tạo\nNguồn: https://vi.wikipedia.org/wiki/Tr%C3%AD_tu%E1%BB%87_nh%C3%A2n_t%E1%BA%A1o\nNội dung:\nTrí tuệ nhân tạo (tiếng Anh: Artificial intelligence, viết tắt là AI) là khả năng của các hệ thống máy tính thực hiện các nhiệm vụ liên quan đến trí thông minh của con người, như học tập, suy luận, giải quyết vấn đề, nhận thức và đưa ra quyết định. Đây là một lĩnh vực nghiên cứu thuộc khoa học máy tính tập trung phát triển và nghiên cứu các phương pháp và phần mềm giúp máy móc có thể nhận thức môi trường xung quanh, cũng như sử dụng học tập và trí tuệ để thực hiện hành động nhằm tối đa hóa khả năng đạt được các mục tiêu đã định.\nCác ứng dụng nổi bật của AI bao gồm công cụ tìm kiếm web tiên tiến (ví dụ: Google Tìm kiếm); hệ thống đề xuất (được sử dụng bởi YouTube, Amazon và Netflix); trợ lý ảo (ví dụ: Trợ lý Google, Siri và Alexa); xe tự lái (ví dụ: Waymo); công cụ sáng tạo và nội dung tạo sinh (ví dụ: mô hình ngôn ngữ và nghệ thuật AI); cùng khả năng ch

In [35]:
# Automatic routing test with explicit source requests
routing_cases = [
    ("Chỉ dùng Wikipedia để tìm thông tin về Hà Nội.", "search_wikipedia"),
    ("Chỉ dùng arXiv để tìm nghiên cứu về large language model agents.", "search_arxiv"),
    ("Hãy đọc URL https://example.com bằng công cụ tải trang web.", "load_web_page"),
]
for prompt, expected_name in routing_cases:
    routed = model_with_tools.invoke(prompt)
    assert routed.tool_calls, f"No tool call for: {prompt}"
    assert routed.tool_calls[0]["name"] == expected_name, routed.tool_calls
    print(f"PASS: automatic routing -> {expected_name}")

PASS: automatic routing -> search_wikipedia
PASS: automatic routing -> search_arxiv
PASS: automatic routing -> load_web_page
